In [ ]:
!pip install pyreadstat --break-system-packages

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
import pandas as pd
import pyreadstat
import warnings
warnings.filterwarnings('ignore')
import os
os.chdir(r"C:\Users\Hp\Downloads\Project 2026 DS")

In [ ]:
_,meta16=pyreadstat.read_sav(r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data\surveydata1617.sav",metadataonly=True)

cl_lbl16=pd.DataFrame({'column': meta16.column_names,'label': meta16.column_labels})
print(cl_lbl16.to_string())

In [ ]:
msoa_check=cl_lbl16[cl_lbl16['column'].str.lower().str.contains('msoa',na=False) |cl_lbl16['label'].str.lower().str.contains('msoa',na=False)]
print(msoa_check.to_string())

In [ ]:

cols_1617=['serial','wt_final','Reg9','LA','LondInOut','Age9','Gend3','Eth7','IMD10','Disab3',
    'NSSEC5','Educ6','Orient4','Relig7','ChildAgeU13','Maternity_pop',
    'Filter_Act','Filter_InsAct','Filter_Inact']

df16,meta16=pyreadstat.read_sav(r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data\surveydata1617.sav",
    usecols=cols_1617,apply_value_formats=True)

print(df16.shape)
print(df16['Reg9'].value_counts())

In [ ]:
lon16=df16[df16['LA'].str.startswith('E09')].copy()
if pd.api.types.is_categorical_dtype(lon16['LA']):
    lon16['LA']=lon16['LA'].cat.remove_unused_categories()
print(f"london respondents 2016-17: {len(lon16)}")
print(f"boroughs: {lon16['LA'].nunique()}")

In [ ]:
print(lon16['Age9'].unique())

['16-24', '35-44', '45-54', '55-64', '25-34', '65-74', '75-84', '85+', NaN]
Categories (8, object): ['16-24', '25-34', '35-44', '45-54', '55-64', '65-74', '75-84', '85+']


In [ ]:
both_missing=lon16[lon16['Orient4'].isna() & lon16['Relig7'].isna()]
print(f"Missing both: {len(both_missing)}")

In [ ]:
both_missing=lon16[lon16['Orient4'].isna() & lon16['Relig7'].isna()]
print(f"Missing both: {len(both_missing)}")
print(f"Missing Orient4 only: {lon16['Orient4'].isna().sum() - len(both_missing)}")
print(f"Missing Relig7 only: {lon16['Relig7'].isna().sum() - len(both_missing)}")

In [ ]:
age9_only=lon16.groupby(['LA','Age9']).apply(lambda x: pd.Series({'pct_active': (x['Filter_Act'] * x['wt_final']).sum() / x['wt_final'].sum() * 100,
    'respondents': len(x)})).reset_index()

print(age9_only[age9_only['Age9'] == 'Not asked / Not applicable'])
print(age9_only['Age9'].unique())

In [ ]:
print(lon16['Filter_Act'].dtype)
print(lon16['Filter_InsAct'].dtype)
print(lon16['Filter_Inact'].dtype)
print(lon16['wt_final'].dtype)

float64
float64
float64
float64


In [ ]:

print(lon16['LA'].dtype)
print(lon16['IMD10'].dtype)
print(lon16['NSSEC5'].dtype)
print(lon16['Eth7'].dtype)

category
category
category
category


In [ ]:
group_cols=['LA','Age9','Gend3','Eth7','IMD10','Disab3','LondInOut','NSSEC5','Educ6','Orient4','Relig7','ChildAgeU13','Maternity_pop']

#making it normal text
for col in group_cols:
    lon16[col]=lon16[col].astype(object).fillna('Not asked / Not applicable').astype(str)

imd_raw16,_=pyreadstat.read_sav(r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data\surveydata1617.sav",usecols=['serial','IMD10'],
    apply_value_formats=False)
imd_raw16=imd_raw16.rename(columns={'IMD10': 'IMD10_raw'})

lon16=lon16.merge(imd_raw16,on='serial',how='left')

imd_map={i: f"Decile {i}" for i in range(1,11)}
imd_map[1]="Decile 1 (most deprived)"
imd_map[10]="Decile 10 (least deprived)"

lon16['IMD10_numeric']=lon16['IMD10_raw']
lon16['IMD10']=lon16['IMD10_raw'].map(imd_map).fillna('Not asked / Not applicable')
lon16=lon16.drop(columns=['IMD10_raw'])

lon16['Filter_Act']=lon16['Filter_Act'].astype(float)
lon16['Filter_InsAct']=lon16['Filter_InsAct'].astype(float)
lon16['Filter_Inact']=lon16['Filter_Inact'].astype(float)
lon16['wt_final']=lon16['wt_final'].astype(float)

print(lon16.dtypes)
print(lon16['IMD10'].value_counts())

In [ ]:
check_cols=['Age9','Gend3','Eth7','IMD10','Disab3','NSSEC5','Educ6','Orient4','Relig7','ChildAgeU13','Maternity_pop']

for col in check_cols:
    print(f"{col} ")
    print(lon16[col].value_counts())
    print()

In [ ]:
def weighted_activity(df,group_col):
    result=df.groupby(group_col).apply(lambda x: pd.Series({ 'pct_active': (x['Filter_Act'] * x['wt_final']).sum() / x['wt_final'].sum() * 100,
        'pct_fairly_active': (x['Filter_InsAct'] * x['wt_final']).sum() / x['wt_final'].sum() * 100,'pct_inactive': (x['Filter_Inact'] * x['wt_final']).sum() / x['wt_final'].sum() * 100,
        'respondents': len(x),'weighted_base': x['wt_final'].sum()})).reset_index()
    return result

In [ ]:
#gap score on each borough
gap16=weighted_activity(lon16,'LA')
gap16=gap16.rename(columns={'LA': 'borough'})
gap16['survey_year']='2016-17'

gap16=gap16[['survey_year','borough','pct_active','pct_fairly_active','pct_inactive','respondents','weighted_base']]

print(gap16.to_string())

In [ ]:
#population profile
demo_cols=['Age9','Gend3','Eth7','IMD10','Disab3','LondInOut','NSSEC5','Educ6','Orient4','Relig7','ChildAgeU13','Maternity_pop']

rows16=[]
for col in demo_cols:
    temp=lon16.groupby(['LA',col]).apply(lambda x: pd.Series({
        'pct_active': (x['Filter_Act'] * x['wt_final']).sum() / x['wt_final'].sum() * 100,
        'pct_fairly_active': (x['Filter_InsAct'] * x['wt_final']).sum() / x['wt_final'].sum() * 100,  'pct_inactive': (x['Filter_Inact'] * x['wt_final']).sum() / x['wt_final'].sum() * 100,
        'respondents': len(x),'weighted_base': x['wt_final'].sum()})).reset_index()

    temp=temp.rename(columns={'LA': 'borough',col: 'category'})
    temp['demographic_group']=col
    rows16.append(temp)

profile16=pd.concat(rows16,ignore_index=True)
profile16['survey_year']='2016-17'
profile16['suppress']=profile16['respondents'] < 30
profile16=profile16[['survey_year','borough','demographic_group','category' 'pct_active','pct_fairly_active','pct_inactive','respondents','weighted_base','suppress']]
print(profile16.head(20).to_string())

In [ ]:
_,meta18=pyreadstat.read_sav(r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data\surveydata1718.sav",metadataonly=True)

cl_lbl18=pd.DataFrame({'column': meta18.column_names,'label': meta18.column_labels})
print(cl_lbl18.to_string())

In [ ]:
cols_1718=['serial','wt_final','Reg9','LA','LondInOut','Age9','Gend3','Eth7','IMD10','Disab3','NSSEC5','Educ6','Orient4','Relig7',
    'ChildAgeU13','Maternity_pop','Filter_Act','Filter_InsAct','Filter_Inact']
df18,meta18=pyreadstat.read_sav(r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data\surveydata1718.sav",
    usecols=cols_1718,apply_value_formats=True)

print(df18.shape)
print(df18['Reg9'].value_counts())

In [ ]:
#only london borough
lon18=df18[df18['LA'].str.startswith('E09')].copy()

if pd.api.types.is_categorical_dtype(lon18['LA']):
    lon18['LA']=lon18['LA'].cat.remove_unused_categories()

print(f"london respondents 2017-18: {len(lon18)}")
print(f"boroughs: {lon18['LA'].nunique()}")

In [ ]:

group_cols=['LA','Age9','Gend3','Eth7','IMD10','Disab3''LondInOut','NSSEC5','Educ6','Orient4','Relig7','ChildAgeU13','Maternity_pop']

for col in group_cols:
    n_missing=lon18[col].isna().sum()
    print(f"{col}: {n_missing} missing")

In [ ]:
for col in group_cols:
    lon18[col]=lon18[col].astype(object).fillna('Not asked / Not applicable').astype(str)

imd_raw18,_=pyreadstat.read_sav(r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data\surveydata1718.sav",
    usecols=['serial','IMD10'],apply_value_formats=False)
imd_raw18=imd_raw18.rename(columns={'IMD10': 'IMD10_raw'})

lon18=lon18.merge(imd_raw18,on='serial',how='left')

lon18['IMD10_numeric']=lon18['IMD10_raw']
lon18['IMD10']=lon18['IMD10_raw'].map(imd_map).fillna('Not asked / Not applicable')
lon18=lon18.drop(columns=['IMD10_raw'])

lon18['Filter_Act']=lon18['Filter_Act'].astype(float)
lon18['Filter_InsAct']=lon18['Filter_InsAct'].astype(float)
lon18['Filter_Inact']=lon18['Filter_Inact'].astype(float)
lon18['wt_final']=lon18['wt_final'].astype(float)

print(lon18.dtypes)
print(lon18['IMD10'].value_counts())

In [ ]:
#2017-18 gap table
gap18=weighted_activity(lon18,'LA')
gap18=gap18.rename(columns={'LA': 'borough'})
gap18['survey_year']='2017-18'

gap18=gap18[['survey_year','borough','pct_active','pct_fairly_active',
               'pct_inactive','respondents','weighted_base']]

print(gap18.to_string())

In [ ]:
#2017-2018 population profile
rows18=[]

for col in demo_cols:
    temp=lon18.groupby(['LA',col]).apply(lambda x: pd.Series({'pct_active': (x['Filter_Act'] * x['wt_final']).sum() / x['wt_final'].sum() * 100,
        'pct_fairly_active': (x['Filter_InsAct'] * x['wt_final']).sum() / x['wt_final'].sum() * 100, 'pct_inactive': (x['Filter_Inact'] * x['wt_final']).sum() / x['wt_final'].sum() * 100,
        'respondents': len(x),'weighted_base': x['wt_final'].sum()})).reset_index()

    temp=temp.rename(columns={'LA': 'borough',col: 'category'})
    temp['demographic_group']=col
    rows18.append(temp)

profile18=pd.concat(rows18,ignore_index=True)
profile18['survey_year']='2017-18'
profile18['suppress']=profile18['respondents'] < 30
profile18=profile18[['survey_year','borough','demographic_group','category','pct_active','pct_fairly_active','pct_inactive' 'respondents','weighted_base','suppress']]

print(profile18.head(20).to_string())

In [ ]:
gap_final=pd.concat([gap16,gap18],ignore_index=True)
profile_final=pd.concat([profile16,profile18],ignore_index=True)
gap_final.to_csv( r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data\gapscore.csv",index=False)

profile_final.to_csv( r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data\popl_profile.csv",index=False)

print("Saved 2 files:")
print(f"gap score table: {gap_final.shape}")
print(f"population profile table: {profile_final.shape}")